# Build Raw-Post FAISS Index

Embeds raw posts (no profile fusion) and saves to `data/vector_db/{dataset_name}_rawpost/`.

```
vector = normalise( embed(raw_post) )
```

**Output per dataset:**
- `data/vector_db/{dataset_name}_rawpost/vectors.faiss`
- `data/vector_db/{dataset_name}_rawpost/vectors_meta.jsonl`

**Anti-leakage:** Only the train split is embedded into the index.

## 1. CONFIG

In [ ]:
import os, sys
from pathlib import Path

# ════════════════════════════════════════════════════════════════════════════
# PROJECT ROOT
# ════════════════════════════════════════════════════════════════════════════
PROJECT_ROOT = str(Path(os.getcwd()).parent.parent)  # notebook/gpt/ -> root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ════════════════════════════════════════════════════════════════════════════
# DATASET — change to 'myp' for the mypersonality dataset
# ════════════════════════════════════════════════════════════════════════════
DATASET_NAME = "essays"   # 'essays'  |  'myp'

# ════════════════════════════════════════════════════════════════════════════
# DERIVED PATHS  (no edits needed below this line for standard layout)
# ════════════════════════════════════════════════════════════════════════════
TRAIN_CSV  = os.path.join(PROJECT_ROOT, "data", "split", DATASET_NAME, "train.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "vector_db", f"{DATASET_NAME}_rawpost")

# ════════════════════════════════════════════════════════════════════════════
# EMBEDDING MODEL
# ════════════════════════════════════════════════════════════════════════════
EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v1.5"
MAX_SEQ_LENGTH  = 2048
FORCE_REBUILD   = False   # set True to overwrite an existing index

# ════════════════════════════════════════════════════════════════════════════
# Path check
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("PATH CHECK")
print("=" * 60)
for label, p in [
    ("PROJECT_ROOT", PROJECT_ROOT),
    ("TRAIN_CSV",    TRAIN_CSV),
    ("OUTPUT_DIR",   OUTPUT_DIR),
]:
    ok = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:20s}: [{ok}]  {p}")

print(f"\n  Dataset         : {DATASET_NAME}")
print(f"  Embedding model : {EMBEDDING_MODEL}")
print(f"  MAX_SEQ_LENGTH  : {MAX_SEQ_LENGTH}")

## 2. Imports

In [ ]:
import json
import time
import numpy as np
import pandas as pd

import rag.embedder as _embedder
_embedder.FALLBACK_EMBEDDING_MODEL = EMBEDDING_MODEL
_embedder.MAX_SEQ_LENGTH           = MAX_SEQ_LENGTH

from rag.embedder    import _embed_single
from rag.faiss_index import FAISSIndex

TRAIT_MAP = {
    "cEXT": "Extraversion",
    "cNEU": "Neuroticism",
    "cAGR": "Agreeableness",
    "cCON": "Conscientiousness",
    "cOPN": "Openness to Experience",
}

print("Imports OK")

## 3. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"Train rows : {len(train_df)}")
print(f"Columns    : {train_df.columns.tolist()}")

def extract_trait_labels(row):
    labels = {}
    for col, name in TRAIT_MAP.items():
        if col not in row.index:
            continue
        raw = str(row[col]).strip().lower()
        if raw in ("high", "low"):
            labels[name] = raw
        elif raw in ("1", "1.0"):
            labels[name] = "high"
        elif raw in ("0", "0.0"):
            labels[name] = "low"
    return labels

print("\nLabel distribution:")
for col, name in TRAIT_MAP.items():
    if col not in train_df.columns:
        continue
    v = train_df[col].astype(str).str.strip().str.lower()
    hi = ((v == "high") | (v == "1") | (v == "1.0")).sum()
    lo = ((v == "low")  | (v == "0") | (v == "0.0")).sum()
    print(f"  {name:25s}: high={hi:5d}, low={lo:5d}")

## 4. Token Length Audit

In [ ]:
from rag.embedder import _get_model
model = _get_model()
tok   = model.tokenizer

EMBED_PREFIX = "search_document: "

lengths = []
for _, row in train_df.iterrows():
    ids = tok.encode(EMBED_PREFIX + str(row["text"]), add_special_tokens=True, truncation=False)
    lengths.append(len(ids))

lengths = np.array(lengths)
print(f"Token lengths (n={len(lengths)}):")
for pct in [50, 75, 90, 95, 99]:
    print(f"  p{pct:2d} = {int(np.percentile(lengths, pct))}")
print(f"  max  = {lengths.max()}")
over = (lengths > MAX_SEQ_LENGTH).sum()
print(f"  > {MAX_SEQ_LENGTH}: {over} ({over / len(lengths) * 100:.1f}%) — sliding-window will be used")

## 5. Embed & Build Index

In [ ]:
index_path = os.path.join(OUTPUT_DIR, "vectors.faiss")
meta_path  = os.path.join(OUTPUT_DIR, "vectors_meta.jsonl")

if not FORCE_REBUILD and os.path.exists(index_path) and os.path.exists(meta_path):
    print(f"Index already exists at {OUTPUT_DIR!r}. Set FORCE_REBUILD=True to overwrite.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    n  = len(train_df)
    t0 = time.time()

    embeddings = []
    meta       = []

    print(f"Embedding {n} raw posts ...")
    for i, (_, row) in enumerate(train_df.iterrows()):
        raw_text = str(row["text"])
        vec      = _embed_single(raw_text)   # shape (768,) float32, L2-normalised
        embeddings.append(vec)

        user_id      = row.get("user_id", f"user_{i}") if "user_id" in row.index else f"user_{i}"
        trait_labels = extract_trait_labels(row)
        meta.append({
            "user_id":      str(user_id),
            "posts_raw":    raw_text,
            "trait_labels": trait_labels,
        })

        if (i + 1) % 200 == 0:
            elapsed = time.time() - t0
            eta     = (n - i - 1) / ((i + 1) / elapsed)
            print(f"  {i + 1}/{n}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

    elapsed = time.time() - t0
    print(f"Embedded {n} posts in {elapsed:.1f}s")

    emb_matrix = np.array(embeddings, dtype="float32")
    idx = FAISSIndex(dimension=emb_matrix.shape[1])
    idx.build(emb_matrix, meta)
    idx.save(index_path, meta_path)
    print(f"Saved -> {OUTPUT_DIR}/  ({n} vectors, dim={emb_matrix.shape[1]})")

## 6. Inspect Output

In [ ]:
print(f"Files in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:35s} {size_mb:>8.2f} MB")

print("\nFirst 3 meta entries:")
with open(meta_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        entry = json.loads(line)
        print(f"  [{i}] user_id={entry['user_id']}  labels={entry['trait_labels']}")
        print(f"       text={entry['posts_raw'][:80]!r}...")

## 7. Sanity Check — Retrieval Accuracy on Test Split

Encode test posts, retrieve top-K from the train index, majority-vote label.

In [ ]:
import faiss as _faiss
from sklearn.metrics import accuracy_score, f1_score

TEST_CSV = os.path.join(PROJECT_ROOT, "data", "split", DATASET_NAME, "test.csv")
TOP_K    = 5

if not Path(TEST_CSV).exists():
    print(f"Test CSV not found: {TEST_CSV}")
else:
    test_df = pd.read_csv(TEST_CSV)
    print(f"Test rows: {len(test_df)}")

    train_index = _faiss.read_index(index_path)
    train_meta  = []
    with open(meta_path, encoding="utf-8") as f:
        for line in f:
            train_meta.append(json.loads(line))

    print("Encoding test posts...")
    t0        = time.time()
    test_embs = []
    for i, (_, row) in enumerate(test_df.iterrows()):
        vec = _embed_single(str(row["text"]))
        test_embs.append(vec)
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(test_df)}")

    test_embs = np.array(test_embs, dtype="float32")
    print(f"Encoded {len(test_embs)} test posts in {time.time() - t0:.1f}s")

    _, neighbor_idx = train_index.search(test_embs, TOP_K)

    TRAIT_FULL = {
        "cOPN": "Openness to Experience",
        "cCON": "Conscientiousness",
        "cEXT": "Extraversion",
        "cAGR": "Agreeableness",
        "cNEU": "Neuroticism",
    }

    print(f"\n=== Retrieval accuracy (top-{TOP_K} majority vote) ===")
    print(f"{'Trait':25s} {'Acc':>8s}  {'Macro-F1':>10s}")
    print("-" * 48)

    for trait_col, trait_full in TRAIT_FULL.items():
        if trait_col not in test_df.columns:
            continue
        gt_labels, preds = [], []
        for i, (_, row) in enumerate(test_df.iterrows()):
            gt = str(row[trait_col]).strip().lower()
            if gt in ("1", "1.0"):  gt = "high"
            if gt in ("0", "0.0"):  gt = "low"
            gt_labels.append(gt)

            nb_labels = [
                train_meta[idx]["trait_labels"].get(trait_full, "low")
                for idx in neighbor_idx[i] if idx >= 0
            ]
            preds.append("high" if nb_labels.count("high") > nb_labels.count("low") else "low")

        acc = accuracy_score(gt_labels, preds)
        f1  = f1_score(gt_labels, preds, average="macro")
        print(f"  {trait_full:25s}  {acc:>8.4f}  {f1:>10.4f}")

## 8. Quick Retrieval Demo — RawPostRetriever

In [ ]:
from rag.raw_post_retriever import RawPostRetriever

retriever = RawPostRetriever(db_dir=OUTPUT_DIR)

SAMPLE_TEXT = str(train_df.iloc[0]["text"])[:300]

for trait in ["Openness to Experience", "Conscientiousness"]:
    results = retriever.retrieve(SAMPLE_TEXT, trait=trait, top_k=3)
    print(f"\n--- {trait} ---")
    for r in results:
        print(f"  [{r['label']}] dist={r['distance']:.4f}  {r['posts_raw'][:80]!r}...")